# SawitGuard-GNN — Lapisan 1 (demonstrator jujur, data yang ada)

Pipeline kanonik Lapisan 1 pada **dataset B** (2.303 tile UAV nadir RGB, 1024², biner
Healthy/Unhealthy). **Bukan BSR** — label kesehatan-tajuk generik (lihat `../layer1_data_audit/AUDIT_REPORT.md`).

**Disiplin:** semua evaluasi memakai **block-CV leave-one-ortho-out** (3 ortomosaik), bukan split
acak (bocor 100%). Metrik jujur: **PR-AUC / ROC-AUC** (kesehatan, kelas timpang) dan **mAP** (deteksi).

Jalankan notebook ini **dari dalam folder `layer1_build/`** (butuh `grids.py` di sampingnya).
Prasyarat: `pip install roboflow ultralytics lightgbm scikit-image opencv-python` + torch CUDA;
set `ROBOFLOW_API_KEY`. **Output tersimpan di file ini kosong — kamu yang menjalankan.**

In [ ]:
import os, warnings, json, re, collections, time
warnings.filterwarnings("ignore")
import numpy as np
import grids
from grids import load_coco
BASE = os.path.dirname(os.path.abspath(grids.__file__))   # = folder layer1_build (robust thd cwd)
CLS = {"Healthy": 0, "Unhealthy": 1}
def region(path):
    m = re.match(r"(\d+)_(\d+)_", os.path.basename(path).split(".rf.")[0])
    return f"{m.group(1)}_{m.group(2)}" if m else "OTHER"
print("BASE =", BASE)

## 0. Unduh dataset B (dilewati bila sudah ada)

In [ ]:
if not os.path.isdir(os.path.join(BASE, "ds_B")):
    from roboflow import Roboflow
    assert os.environ.get("ROBOFLOW_API_KEY"), "set ROBOFLOW_API_KEY dulu"
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    (rf.workspace("health-detection").project("oil-palm-health-detection")
       .version(2).download("coco", location=os.path.join(BASE, "ds_B"), overwrite=True))
else:
    print("ds_B sudah ada")

cats, anns = load_coco("B")
by_img = collections.defaultdict(list)
for a in anns:
    by_img[a["path"]].append(a)
print(f"tajuk(anotasi)={len(anns)}  gambar={len(by_img)}  kelas={set(a['cat'] for a in anns)}")

## Tahap 2 — Klasifikasi kesehatan per-pohon (LightGBM atas fitur tajuk)
Fitur per-tajuk (luas via ExG, warna/greenness, tekstur) → LightGBM → block-CV leave-one-ortho-out.

In [ ]:
import cv2
def crown_features(c):
    R, G, B = c[..., 0], c[..., 1], c[..., 2]
    exg = 2*G - R - B
    gray = c.mean(2)
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    gmag = np.sqrt(gx*gx + gy*gy); denom = R + G + B + 1e-6
    return dict(crown_cov=float((exg > 0).mean()),
        R_mean=R.mean(), G_mean=G.mean(), B_mean=B.mean(),
        R_std=R.std(), G_std=G.std(), B_std=B.std(),
        exg_mean=exg.mean(), exg_std=exg.std(),
        GmR=(G-R).mean(), GmB=(G-B).mean(), green_frac=(G/denom).mean(),
        lap_var=float(lap.var()), gmag_mean=float(gmag.mean()), gmag_std=float(gmag.std()),
        bright=gray.mean())

rows = []; t0 = time.time()
for pth, alist in by_img.items():
    im = cv2.imread(pth)
    if im is None: continue
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32)
    H, W = im.shape[:2]; reg = region(pth)
    for a in alist:
        if a["cat"] not in CLS: continue
        x, y, w, h = a["bbox"]
        x0, y0 = max(0, int(x)), max(0, int(y)); x1, y1 = min(W, int(x+w)), min(H, int(y+h))
        if x1-x0 < 3 or y1-y0 < 3: continue
        f = crown_features(im[y0:y1, x0:x1])
        f.update(box_w=w, box_h=h, box_area=w*h, aspect=w/(h+1e-6))
        rows.append((f, CLS[a["cat"]], reg))
feat_names = list(rows[0][0].keys())
X = np.array([[r[0][k] for k in feat_names] for r in rows], dtype=np.float32)
y = np.array([r[1] for r in rows]); regs = np.array([r[2] for r in rows])
print(f"tajuk={len(y)}  positif(Unhealthy)={int(y.sum())}  base_rate={100*y.mean():.2f}%  "
      f"ekstraksi={time.time()-t0:.0f}s")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
uregs = sorted(set(regs))
params = dict(objective="binary", verbose=-1, num_leaves=31, learning_rate=0.05,
              feature_fraction=0.8, min_data_in_leaf=50, seed=42)
prauc, roc, f1 = [], [], []
for held in uregs:
    tr, te = regs != held, regs == held
    m = lgb.train(params, lgb.Dataset(X[tr], label=y[tr]), num_boost_round=300)
    p = m.predict(X[te])
    prauc.append(average_precision_score(y[te], p)); roc.append(roc_auc_score(y[te], p))
    f1.append(f1_score(y[te], (p > 0.5).astype(int), zero_division=0))
    print(f"  hold {held}: PR-AUC={prauc[-1]:.3f}  ROC-AUC={roc[-1]:.3f}  (n={int(te.sum())}, pos={int(y[te].sum())})")
print(f"\nKESEHATAN  PR-AUC = {np.mean(prauc):.3f} +/- {np.std(prauc):.3f}   "
      f"ROC-AUC = {np.mean(roc):.3f}   (acak PR-AUC = base rate {y.mean():.4f})")
# konteks fitur (bukan klaim)
mm = lgb.train(params, lgb.Dataset(X, label=y), num_boost_round=300)
print("fitur teratas:", [k for k, _ in sorted(zip(feat_names, mm.feature_importance()), key=lambda t: -t[1])[:6]])

> **Catatan jujur:** satu ortomosaik biasanya kolaps (PR-AUC jauh lebih rendah) → variasi antar-situs
> besar = bukti langsung batas generalisasi data satu-situs. Perbedaan kecil (< ~1 std antar-fold) = noise.

## Tahap 1 — Deteksi tajuk (YOLO11n, leave-one-ortho-out)
Konversi ke format YOLO dengan fold per-ortomosaik, lalu latih.

In [ ]:
import shutil
def yolo_prep():
    ROOT = os.path.join(BASE, "yolo_B"); IMG = os.path.join(ROOT, "images"); LAB = os.path.join(ROOT, "labels")
    os.makedirs(IMG, exist_ok=True); os.makedirs(LAB, exist_ok=True)
    img_region = {}
    for pth, alist in by_img.items():
        fn = os.path.basename(pth); stem = os.path.splitext(fn)[0]; W, H = alist[0]["W"], alist[0]["H"]
        dst = os.path.join(IMG, fn)
        if not os.path.exists(dst):
            try: os.link(pth, dst)
            except Exception: shutil.copy(pth, dst)
        lines = [f"{CLS[a['cat']]} {(a['bbox'][0]+a['bbox'][2]/2)/W:.6f} {(a['bbox'][1]+a['bbox'][3]/2)/H:.6f} "
                 f"{a['bbox'][2]/W:.6f} {a['bbox'][3]/H:.6f}" for a in alist if a["cat"] in CLS]
        open(os.path.join(LAB, stem + ".txt"), "w").write("\n".join(lines))
        img_region[dst] = region(fn)
    regions = sorted(set(img_region.values()))
    for i, held in enumerate(regions):
        tr = [p for p, r in img_region.items() if r != held]; va = [p for p, r in img_region.items() if r == held]
        open(os.path.join(ROOT, f"fold{i}_train.txt"), "w").write("\n".join(tr))
        open(os.path.join(ROOT, f"fold{i}_val.txt"), "w").write("\n".join(va))
        open(os.path.join(ROOT, f"fold{i}.yaml"), "w").write(
            f"path: {ROOT}\ntrain: fold{i}_train.txt\nval: fold{i}_val.txt\nnames:\n  0: Healthy\n  1: Unhealthy\n")
    return ROOT, regions
YOLO_ROOT, YOLO_REGIONS = yolo_prep()
print("fold (per ortomosaik):", YOLO_REGIONS)

In [ ]:
# Di notebook Windows pakai workers=0 (hindari masalah spawn dataloader).
# Untuk full run yang cepat, lebih baik jalankan skrip: FOLDS=0,1,2 EPOCHS=50 IMGSZ=640 python yolo_train.py
import torch
from ultralytics import YOLO
EPOCHS, IMGSZ, FOLDS = 15, 512, [0]      # PAPER: EPOCHS=50, IMGSZ=640, FOLDS=[0,1,2]
DEV = 0 if torch.cuda.is_available() else "cpu"
yres = {}
for i in FOLDS:
    m = YOLO("yolo11n.pt")
    m.train(data=os.path.join(YOLO_ROOT, f"fold{i}.yaml"), epochs=EPOCHS, imgsz=IMGSZ, device=DEV,
            batch=16, cache="ram", workers=0, project=os.path.join(BASE, "yolo_runs"),
            name=f"fold{i}", exist_ok=True, verbose=False, plots=False, seed=42)
    mt = m.val(data=os.path.join(YOLO_ROOT, f"fold{i}.yaml"), device=DEV, verbose=False, plots=False)
    yres[i] = dict(map50=float(mt.box.map50), map=float(mt.box.map), mp=float(mt.box.mp), mr=float(mt.box.mr))
    print(f"fold{i}: mAP50={yres[i]['map50']:.3f}  mAP50-95={yres[i]['map']:.3f}  "
          f"P={yres[i]['mp']:.3f}  R={yres[i]['mr']:.3f}")
if len(FOLDS) > 1:
    ms = [yres[i]["map50"] for i in FOLDS]
    print(f"\nDETEKSI mAP50 = {np.mean(ms):.3f} +/- {np.std(ms):.3f}")

## Tahap 3 — Segmentasi luas tajuk (kualitatif, ExG+Otsu; TANPA klaim IoU)

In [ ]:
import random
from PIL import Image
random.seed(3)
big = [a for a in anns if a["cat"] == "Healthy" and (a["bbox"][2]**2 + a["bbox"][3]**2)**0.5 > 120]
sample = random.sample(big, 6); cell = 180
canvas = Image.new("RGB", (2*cell*3, cell*2), (15, 15, 15))
for k, a in enumerate(sample):
    im = cv2.cvtColor(cv2.imread(a["path"]), cv2.COLOR_BGR2RGB).astype(np.float32)
    x, y0, w, h = [int(v) for v in a["bbox"]]
    c = im[max(0,y0):y0+h, max(0,x):x+w]
    exg = 2*c[...,1]-c[...,0]-c[...,2]
    e8 = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX).astype("uint8")
    _, mask = cv2.threshold(e8, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    r, cc = divmod(k, 3)
    canvas.paste(Image.fromarray(c.astype("uint8")).resize((cell,cell)), (cc*cell*2, r*cell))
    canvas.paste(Image.fromarray(mask).convert("RGB").resize((cell,cell)), (cc*cell*2+cell, r*cell))
canvas.save(os.path.join(BASE, "crown_area_demo.jpg"))
print("disimpan crown_area_demo.jpg — kiri=crop, kanan=mask ExG+Otsu (luas tajuk OK; pelepah tak terpisah)")
canvas

## Ringkasan & klaim maksimum jujur
- **Deteksi tajuk** (YOLO11n, ortho held-out): mAP50 ~0,76 → tahap deteksi Lapisan 1 layak.
- **Kesehatan per-pohon** (LightGBM, block-CV): ROC-AUC ~0,84, PR-AUC ~0,13 (~9x di atas acak),
  **dengan variasi antar-situs besar** (satu ortomosaik kolaps).
- **Luas tajuk**: ExG kualitatif; **tanpa IoU** (butuh mask GT Sembawa — future work).

> Klaim maksimum: *"deteksi + penilaian-kesehatan-kasar tajuk sawit per-pohon dari UAV RGB, satu-situs;
> label kesehatan generik, BUKAN BSR."* Validasi BSR per-pohon (Sembawa) = pekerjaan lanjutan.